# Week 5 — Monday: Clustering

**DATA 202 · Calvin University**

**180 music listeners**, two numbers each: hours of music per week, and number of different artists.

> 🎯 **Nobody told us what "kinds" of listeners exist. Can we find the groups anyway?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Look at the data |
| ~15 min | 2 · A recipe for finding groups: k-means (SLO 05A) |
| ~10 min | 3 · How many groups? (SLO 05B) |
| ~5 min | 4 · Who decides what "similar" means? |
| ~5 min | 5 · Zooming out |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk

---
## 1 · Look at the Data · ~5 min

**listeners** — 180 made-up listeners, one row each:

* `hours_per_week` — hours of music per week
* `distinct_artists` — different artists played

In [ ]:
import pandas as pd
import plotly.express as px

listeners = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/listeners.csv")
print(listeners.shape)
listeners.head()

In [ ]:
fig = px.scatter(listeners, x="hours_per_week", y="distinct_artists",
                 hover_name="listener_id", title="180 Listeners — No Labels")
fig.show()

> 📄 **Handout 1 — How many groups?** Circle every group you see. 🗣️ Compare with a neighbor.

<details><summary>Answer</summary>

Most people see 3 or 4. Is the bottom-left one group or two? No single right answer.
</details>

Your eye took a second. A computer needs a **recipe**.

---
## 2 · A Recipe for Finding Groups: k-means (SLO 05A) · ~15 min

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/05/images/kmeans_animation.gif" width="440" alt="Animation of k-means with six clusters: black X centroids move step by step, and the colored regions around them are redrawn each time, until nothing changes.">

> 💬 **Watch the ✕s** (the **centroids**, the middle of each group). What makes a point change color? What makes a ✕ move?

<details><summary>Answer</summary>

* **Assign** — each point takes the color of its **nearest** ✕
* **Update** — each ✕ moves to the **middle (mean)** of its points
* Repeat until nothing moves
</details>

> 📄 **Handout 2 — k-means by hand** (~5 min). From the four ✕ (**A–D**): **assign** every listener to its nearest ✕, then **update** — draw each group's new ✕ at its middle. 🗣️ Compare with a neighbor, then with *Round 1* below.

In [ ]:
# @title Helper: k-means one round at a time (double-click to see the code)
import numpy as np
from plotly.subplots import make_subplots

def run_kmeans(X, start_rows):
    """Start with the listeners in `start_rows` as centroids, then repeat assign / update."""
    centroids = X[start_rows].astype(float)
    snapshots = [("Start", None, centroids)]
    step = 0
    while True:
        step += 1
        distances = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = distances.argmin(axis=1)                                            # assign: nearest centroid
        new_centroids = np.array([X[labels == j].mean(axis=0) for j in range(len(centroids))])  # update: mean
        if np.allclose(new_centroids, centroids):                                    # nothing moved: done
            snapshots.append((f"Round {step}: nothing moves — done", labels, centroids))
            return snapshots, labels, centroids
        snapshots.append((f"Round {step}, assign", labels, centroids))
        snapshots.append((f"Round {step}, update", labels, new_centroids))
        centroids = new_centroids

COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]   # A blue · B red · C green · D purple

def plot_snapshots(X, snapshots, cols=2):
    """One panel per snapshot. Big black X = centroid."""
    rows = -(-len(snapshots) // cols)
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=[s[0] for s in snapshots],
                        shared_xaxes=True, shared_yaxes=True)
    for i, (title, labels, centroids) in enumerate(snapshots):
        r, c = i // cols + 1, i % cols + 1
        colors = "lightgray" if labels is None else [COLORS[j] for j in labels]
        fig.add_scatter(x=X[:, 0], y=X[:, 1], mode="markers", marker=dict(size=5, color=colors),
                        showlegend=False, row=r, col=c)
        fig.add_scatter(x=centroids[:, 0], y=centroids[:, 1], mode="markers",
                        marker=dict(symbol="x", size=13, color="black", line=dict(width=2, color="white")),
                        showlegend=False, row=r, col=c)
    fig.update_layout(height=300 * rows + 60, margin=dict(t=60, b=20))
    fig.show()

In [ ]:
X = listeners[["hours_per_week", "distinct_artists"]].to_numpy()

snapshots, labels, centroids = run_kmeans(X, start_rows=[38, 71, 129, 158])   # the handout's A, B, C, D
plot_snapshots(X, snapshots[:3] + snapshots[-1:])                             # start, round 1, the end

> 💬 **Question:** Did your groups and your new ✕s match *Round 1*?

<details><summary>Answer</summary>

* **Assign** — the ✕s stay put; the colors change
* **Update** — the colors stay put; the ✕s move (**C** and **D** move most)
* The computer repeats until nothing moves — here, round 6
</details>

---
### That recipe is **k-means**

1. **Start** — pick `k` centroids
2. **Assign** — each point → its nearest centroid
3. **Update** — each centroid → the mean of its points
4. **Repeat** 2–3 until nothing moves

Our choices (**hyperparameters**): `k = 4`, the starting points, straight-line distance.

A different start can end differently → scikit-learn's `KMeans` tries 10 starts (`n_init=10`) and keeps the best.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X)
listeners["cluster"] = kmeans.labels_

fig = px.scatter(listeners, x="hours_per_week", y="distinct_artists",
                 color=listeners["cluster"].astype(str), hover_name="listener_id",
                 title="k-means with k = 4")
fig.update_layout(legend_title_text="cluster")
fig.show()

> 📄 **Handout 3 — Name the groups** (~3 min). Cluster numbers mean nothing. Write a **short name** next to each group; ★ star a listener who could belong to **two** groups. 🗣️ Compare names with a neighbor.

In [ ]:
cluster_summary = listeners.groupby("cluster").agg(
    n_listeners=("listener_id", "count"),
    avg_hours=("hours_per_week", "mean"),
    avg_artists=("distinct_artists", "mean"),
).round(1)
cluster_summary.sort_values("avg_hours")

<details><summary>Possible names</summary>

| avg hours | avg artists | listeners | a possible name |
|---|---|---|---|
| 6.1 | 7.9 | 59 | Light listeners |
| 12.8 | 16.1 | 46 | Everyday listeners |
| 18.0 | 31.9 | 39 | Explorers |
| 28.2 | 6.1 | 36 | Superfans |

The names are *ours* — yours may differ.
</details>

---
## 3 · How Many Groups? (SLO 05B) · ~10 min

We said 4 by eye. Can the data tell us?

**Inertia** = how tight the groups are (sum of squared distances to each centroid)

* Always falls as `k` grows → look for the **elbow**, where more groups stop helping

> 📄 **Handout 4:** circle where you'd put the elbow.

In [ ]:
inertias = []
k_values = range(1, 11)
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

fig = px.line(x=list(k_values), y=inertias, markers=True,
              labels={"x": "Number of Clusters (k)", "y": "Inertia (sum of squared distances)"},
              title="Elbow Method for Listeners")
fig.show()

**Elbow at k = 4** (reading it as 3 isn't crazy — elbows are judgment calls).

**Silhouette score** = is each point much closer to its own group than to the next one?

* −1 (wrong group) … +1 (well matched) — **higher is better**
* Doesn't automatically favor a bigger `k`

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X)
    sil_scores.append(silhouette_score(X, km_labels))

fig = px.line(x=list(range(2, 9)), y=sil_scores, markers=True,
              labels={"x": "Number of Clusters (k)", "y": "Silhouette score"},
              title="Silhouette Score for Listeners")
fig.show()

**Best silhouette: k = 3** (0.62), with k = 4 close behind (0.58).

> 🗣️ **Pair Talk:** Elbow says 4, silhouette says 3. Which do you choose, and why? Write it on Handout 4.

<details><summary>Answer</summary>

Both are defensible: `k = 3` merges light and everyday listeners; `k = 4` keeps them apart. It depends on what the groups are *for* — no metric decides that.
</details>

---
## 4 · Who Decides What "Similar" Means? · ~5 min

k-means returned **0, 1, 2, 3**. The names were ours — and so was a lot more.

> "Resemblance legitimates lumping together, but the basis for establishing proper resemblance is always difficult and often contested."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** Which decisions behind our four groups did a *person* make, not the algorithm?

<details><summary>Answer</summary>

* **Which numbers** — hours and artists (not genre, skips, time of day…)
* **How many groups** — we typed `4`
* **What "close" means** — straight-line distance; one hour = one artist
* **What the groups mean** — the names

Harmless for playlists. Not when the groups decide prices, loans, or jobs.
</details>

---
## 5 · Zooming Out · ~5 min

**Features, but no labels** → **unsupervised learning**: finding patterns with no correct answer to check against. Clustering is one kind; **dimensionality reduction** (Wednesday) is another.

**Kinds of clustering**

| Kind | Idea | Examples |
|---|---|---|
| **Exclusive** | each point in exactly **one** cluster | **k-means**, DBSCAN |
| Overlapping | a point in several clusters, to a degree | Gaussian Mixture Models |
| Hierarchical | clusters nested inside clusters | agglomerative clustering |

Your ★ listener from Handout 3: k-means forced a yes-or-no; an **overlapping** method could say "70% everyday, 30% light".

**Wednesday:** a listener is 2 numbers; a handwritten digit is 64. How do you *look at* data you can't plot? *(PCA, SLO 05C)*